# Synthetic Control File for Employee Records
- Goal: Generate synthetic employee records for testing and development purposes.

## Imports

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date, datetime
from typing import Dict, List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Configs
- This configuration will drive downstream generators (employees -> payroll -> GL).
- Keep config objects typed and deterministic (seeded) for reproducibility.

In [2]:
LocationStrategy = Literal["NY", "SF", "REMOTE"]
JobFamily = Literal["ENGINEERING", "SALES", "PRODUCT", "G&A"]
EmailUniquenessStrategy = Literal["append_id", "dedupe_counter"]

In [3]:
@dataclass(frozen=True)
class CompanyConfig:
    company_name: str = "Gamma Software Solutions, Inc."
    company_domain: str = "gammasoftware.com"
    entity_id: str = "ENT100"
    currency: str = "USD"


@dataclass(frozen=True)
class TimeConfig:
    start_date: date = date(2020, 1, 1)
    as_of_date: date = date(2026, 1, 1)


@dataclass(frozen=True)
class PopulationConfig:
    start_headcount: int = 3
    target_headcount: int = 105


@dataclass(frozen=True)
class IdConfig:
    employee_id_prefix: str = "EMP"
    employee_id_width: int = 6


@dataclass(frozen=True)
class PolicyConfig:
    include_terminations: bool = False
    termination_rate_annual: float = 0.0
    email_uniqueness_strategy: EmailUniquenessStrategy = "append_id"


@dataclass(frozen=True)
class DistributionConfig:
    # high-level organizational job family mix
    job_family_weights: Dict[JobFamily, float] = field(
        default_factory = lambda: {
            "ENGINEERING": 0.5,
            "SALES": 0.15,
            "PRODUCT": 0.15,
            "G&A": 0.2
        }
    )

    # location mix
    location_weights: Dict[LocationStrategy, float] = field(
        default_factory = lambda: {
            "NY": 0.4,
            "SF": 0.3,
            "REMOTE": 0.3
        }
    )

    # job family levels (L1 to L6)
    level_weights_by_family: Dict[JobFamily, Dict[str, float]] = field(
        default_factory = lambda: {
            "ENGINEERING": {"L1": 0.10, "L2": 0.20, "L3": 0.25, "L4": 0.25, "L5": 0.15, "L6": 0.05},
            "SALES": {"L1": 0.15, "L2": 0.25, "L3": 0.25, "L4": 0.20, "L5": 0.10, "L6": 0.05},
            "PRODUCT": {"L1": 0.10, "L2": 0.25, "L3": 0.25, "L4": 0.20, "L5": 0.15, "L6": 0.05},
            "G&A": {"L1": 0.20, "L2": 0.30, "L3": 0.25, "L4": 0.15, "L5": 0.08, "L6": 0.02},
        }
    )


@dataclass(frozen=True)
class RunConfig:
    seed: int = 42
    source: str = "SYNTHETIC_V1"
    created_at: datetime = field(default_factory=lambda: datetime.now())


@dataclass(frozen=True)
class GlobalConfig:
    company: CompanyConfig = field(default_factory=CompanyConfig)
    time: TimeConfig = field(default_factory=TimeConfig)
    population: PopulationConfig = field(default_factory=PopulationConfig)
    ids: IdConfig = field(default_factory=IdConfig)
    policy: PolicyConfig = field(default_factory=PolicyConfig)
    dist: DistributionConfig = field(default_factory=DistributionConfig)
    run: RunConfig = field(default_factory=RunConfig)


CONFIG = GlobalConfig()
CONFIG

GlobalConfig(company=CompanyConfig(company_name='Gamma Software Solutions, Inc.', company_domain='gammasoftware.com', entity_id='ENT100', currency='USD'), time=TimeConfig(start_date=datetime.date(2020, 1, 1), as_of_date=datetime.date(2026, 1, 1)), population=PopulationConfig(start_headcount=3, target_headcount=105), ids=IdConfig(employee_id_prefix='EMP', employee_id_width=6), policy=PolicyConfig(include_terminations=False, termination_rate_annual=0.0, email_uniqueness_strategy='append_id'), dist=DistributionConfig(job_family_weights={'ENGINEERING': 0.5, 'SALES': 0.15, 'PRODUCT': 0.15, 'G&A': 0.2}, location_weights={'NY': 0.4, 'SF': 0.3, 'REMOTE': 0.3}, level_weights_by_family={'ENGINEERING': {'L1': 0.1, 'L2': 0.2, 'L3': 0.25, 'L4': 0.25, 'L5': 0.15, 'L6': 0.05}, 'SALES': {'L1': 0.15, 'L2': 0.25, 'L3': 0.25, 'L4': 0.2, 'L5': 0.1, 'L6': 0.05}, 'PRODUCT': {'L1': 0.1, 'L2': 0.25, 'L3': 0.25, 'L4': 0.2, 'L5': 0.15, 'L6': 0.05}, 'G&A': {'L1': 0.2, 'L2': 0.3, 'L3': 0.25, 'L4': 0.15, 'L5': 0.0

## Schema Definition
- Define the schema for employee records including fields such as employee ID, name, department, role, hire date, and salary
- In productionizing, use Pydantic

In [4]:
# specification of each column in the table schema
@dataclass(frozen=True)
class ColumnSpec:
    name: str
    dtype: str
    nullable: bool = True
    unique: bool = False
    description: str = ""


@dataclass(frozen=True)
class TableSchema:
    name: str
    columns: List[ColumnSpec]
    primary_key: Optional[str] = None


    def column_names(self) -> List[str]:
        return [c.name for c in self.columns]
    
    def dtype_map(self) -> Dict[str, str]:
        return {c.name: c.dtype for c in self.columns}
    
    def unique_columns(self) -> List[str]:
        return [c.name for c in self.columns if c.unique]
    
    def required_columns(self) -> List[str]:
        return [c.name for c in self.columns if not c.nullable]


# Employee table dimensions
DIM_EMPLOYEE_SCHEMA = TableSchema(
    name="dim_employee",
    primary_key="employee_id",
    columns=[
        # employee identity
        ColumnSpec(name="employee_id", dtype="string", nullable=False, unique=True, description="Employee ID (e.g. E1000000)"),
        ColumnSpec(name="first_name", dtype="string", nullable=False, description="Employee first name"),
        ColumnSpec(name="last_name", dtype="string", nullable=False, description="Employee last name"),
        ColumnSpec(name="full_name", dtype="string", nullable=False, description="Employee full name"),
        ColumnSpec(name="email", dtype="string", nullable=False, unique=True, description="Corporate email"),
        ColumnSpec(name="phone", dtype="string", nullable=True, description="Phone number"),
    
        # employment metadata
        ColumnSpec(name="hire_date", dtype="datetime64[ns]", nullable=False, description="Hiring date"),
        ColumnSpec(name="terminatation_date", dtype="datetime64[ns]", nullable=False, description="Termination date (NaT if active)"),
        ColumnSpec(name="employment_status", dtype="string", nullable=False, description="Active or Terminated"),
        ColumnSpec(name="job_family", dtype="string", nullable=False, description="ENGINEERING, SALES, PRODUCT, or G&A"),
        ColumnSpec(name="level", dtype="string", nullable=False, description="L1, L2, ..., L6"),
        ColumnSpec(name="title", dtype="string", nullable=True, description="Optional title based on job family and level"),

        # financial / organizational
        ColumnSpec(name="department", dtype="string", nullable=False, description="Job family unless sub-departments are needed"),
        ColumnSpec(name="cost_center", dtype="string", nullable=False, description="Cost center code"),
        ColumnSpec(name="manager_id", dtype="string", nullable=True, description="Manager of employee; CEO excluded"),
        ColumnSpec(name="location", dtype="string", nullable=False, description="NY, SF, REMOTE"),
        ColumnSpec(name="state", dtype="string", nullable=False, description="State/province proxy for tax treatment"),

        # other metadata
        ColumnSpec("created_at", "datetime64[ns]", nullable=False, description="Generation timestamp (UTC)"),
        ColumnSpec("source", "string", nullable=False, description="Source tag (SYNTHETIC_V1)"),
        ColumnSpec("seed", "int64", nullable=False, description="Seed used for reproducibility"),
    ]
)

DIM_EMPLOYEE_SCHEMA

TableSchema(name='dim_employee', columns=[ColumnSpec(name='employee_id', dtype='string', nullable=False, unique=True, description='Employee ID (e.g. E1000000)'), ColumnSpec(name='first_name', dtype='string', nullable=False, unique=False, description='Employee first name'), ColumnSpec(name='last_name', dtype='string', nullable=False, unique=False, description='Employee last name'), ColumnSpec(name='full_name', dtype='string', nullable=False, unique=False, description='Employee full name'), ColumnSpec(name='email', dtype='string', nullable=False, unique=True, description='Corporate email'), ColumnSpec(name='phone', dtype='string', nullable=True, unique=False, description='Phone number'), ColumnSpec(name='hire_date', dtype='datetime64[ns]', nullable=False, unique=False, description='Hiring date'), ColumnSpec(name='terminatation_date', dtype='datetime64[ns]', nullable=False, unique=False, description='Termination date (NaT if active)'), ColumnSpec(name='employment_status', dtype='string', 

In [5]:
# Summary table of the employee schema
schema_summary = pd.DataFrame(
    {
        "column": [col.name for col in DIM_EMPLOYEE_SCHEMA.columns],
        "dtype": [col.dtype for col in DIM_EMPLOYEE_SCHEMA.columns],
        "nullable": [col.nullable for col in DIM_EMPLOYEE_SCHEMA.columns],
        "unique": [col.unique for col in DIM_EMPLOYEE_SCHEMA.columns],
        "description": [col.description for col in DIM_EMPLOYEE_SCHEMA.columns]
    }
)

schema_summary

,column,dtype,nullable,unique,description
0,employee_id,string,False,True,Employee ID (e.g. E1000000)
1,first_name,string,False,False,Employee first name
2,last_name,string,False,False,Employee last name
3,full_name,string,False,False,Employee full name
4,email,string,False,True,Corporate email
5,phone,string,True,False,Phone number
6,hire_date,datetime64[ns],False,False,Hiring date
7,terminatation_date,datetime64[ns],False,False,Termination date (NaT if active)
8,employment_status,string,False,False,Active or Terminated
9,job_family,string,False,False,"ENGINEERING, SALES, PRODUCT, or G&A"


## Data Generation
- Generate hire dates with growth from starting employee number to ending employee number over a specified date range.
- Create identity fields using `faker`
- Validate constraints and profile outputs


In [6]:
def _rng(seed: int) -> np.random.Generator:
    """
    Return a numpy Generator seeded for reproducible random draws.

    Parameters
    ----------
    seed : int
        Integer seed to initialize the RNG.

    Returns
    -------
    numpy.random.Generator
        A Generator instance initialized with `seed`.
    """
    return np.random.default_rng(seed)

_rng(42)

Generator(PCG64) at 0x198B9FA25E0

### Hire Date Generator
- Function to generate hire dates based on employee number growth over time
    - Assume exponential growth model for employee hires

In [7]:
def generate_hire_dates(
    company_start_date: pd.Timestamp,
    as_of_date: pd.Timestamp,
    initial_headcount: int,
    final_headcount: int,
    rng: np.random.Generator,
    growth_curve: str = "logistic",
    growth_steepness: float = 7.5
) -> pd.DatetimeIndex:
    """
    Generate employee hire dates that model company headcount growth over time.

    The function enforces:
    - `initial_headcount` hires at (or effectively at) company start
    - total headcount reaching `final_headcount` by `as_of_date`
    - a configurable growth shape (linear or backloaded/logistic)

    Parameters
    ----------
    company_start_date : pd.Timestamp
        Date the company was founded / began hiring.
    as_of_date : pd.Timestamp
        Snapshot date by which final_headcount must be reached.
    initial_headcount : int
        Number of employees at company_start_date.
    final_headcount : int
        Total number of employees as of as_of_date.
    rng : np.random.Generator
        Seeded random number generator for reproducibility.
    growth_curve : str
        "linear" for uniform hiring, "logistic" for startup-style backloaded growth.
    growth_steepness : float
        Controls how aggressively hiring is backloaded when using logistic growth.

    Returns
    -------
    pd.DatetimeIndex
        Sorted hire dates of length `final_headcount`.
    """
    # Basic validation
    if final_headcount < initial_headcount:
        raise ValueError("final_headcount must be >= initial_headcount")
    if company_start_date >= as_of_date:
        raise ValueError("company_start_date must be < as_of_date")
    
    # Seed initial hires at the company founding
    remaining_hires = final_headcount - initial_headcount
    founding_hires = [company_start_date] * initial_headcount

    if remaining_hires == 0:
        return pd.DatetimeIndex(founding_hires)
    
    # Hiring window length (in days)
    hiring_window_days = int((as_of_date - company_start_date).days)
    if hiring_window_days <= 0:
        return pd.DatetimeIndex(founding_hires)
    
    # Growth curve handling - linearly spaced dates vs. exponential
    if growth_curve == "linear":
        # Uniform hiring time dates
        uniform_samples = rng.random(remaining_hires)
        day_offsets = (uniform_samples * hiring_window_days).astype(int)
    elif growth_curve == "logistic":
        uniform_samples = rng.random(remaining_hires)
        uniform_samples = np.clip(uniform_samples, 1e-9, 1 - 1e-9)
        logit_space = np.log(uniform_samples / (1 - uniform_samples))
        scaled_curve = 1 / (1 + np.exp(-logit_space / growth_steepness))
        day_offsets = (scaled_curve * hiring_window_days).astype(int)
    else:
        raise ValueError("growth_curve must be 'linear' or 'logistic'")

    # Add a small amount of noise to prevent any clustering
    jitter = rng.integers(low=0, high=3, size=remaining_hires)
    day_offsets = np.clip(day_offsets + jitter, 0, hiring_window_days)

    # Convert offsets to timestamps and return the starting member dates and new hires over time
    growth_hires = [company_start_date + pd.Timedelta(days=int(day)) for day in day_offsets]    
    all_hire_dates = pd.to_datetime(founding_hires + growth_hires).sort_values()
    return pd.DatetimeIndex(all_hire_dates)


# Validate
rng = _rng(CONFIG.run.seed)

hire_dates = generate_hire_dates(
    company_start_date=pd.Timestamp(CONFIG.time.start_date),
    as_of_date=pd.Timestamp(CONFIG.time.as_of_date),
    initial_headcount=CONFIG.population.start_headcount,
    final_headcount=CONFIG.population.target_headcount,
    rng=rng,
    growth_curve="logistic",
    growth_steepness=3
)

hire_dates[-5:]

DatetimeIndex(['2024-03-13', '2024-06-24', '2024-07-17', '2024-07-31',
               '2024-08-25'],
              dtype='datetime64[ns]', freq=None)